In [2]:
import pyperclip as pcl

am_ru_lines = pcl.paste().split("\n")
ru_am = {}
for line in am_ru_lines:
    sp = line.strip().split("\t")
    if sp:
        ru_am[int(sp[1])] = int(sp[0])

In [9]:
am_ru = {v: k for k, v in ru_am.items() if k <= 90}

In [10]:
len(am_ru)

59

In [8]:
import requests
import time

def get_results(id_):
    req = requests.get(f"https://api.rating.chgk.net/tournaments/{id_}/results.json?includeMasksAndControversials=1")
    time.sleep(0.5)
    result = []
    for res in req.json():
        q_num_to_res = {
            i + 1: v for i, v in enumerate(res["mask"])
        }
        result.append({"tournament_id": id_, "team_id": res["team"]["id"], "team_name": res["current"]["name"], "q_num_to_res": q_num_to_res})
    return result


results = {}
for id_ in (11749, 11756, 12169):
    results[id_] = get_results(id_)

In [22]:
import openpyxl

wb = openpyxl.Workbook()

am_roc_chr = wb.create_sheet("ЧА+ЧР+РОК")
roc_chr = wb.create_sheet("ЧР+РОК")

q_keys = [int(x) for x in sorted(am_ru)]
q_keys_two = list(range(1, 91))
header = [
    "Турнир",
    "ID команды",
    "Название",
    "Сумма",
    "№ ЧА"
] + q_keys
am_roc_chr.append(header)
header2 = [
    "",
    "",
    "",
    "",
    "№ ЧР/РОК"
] + [am_ru[x] for x in q_keys]
am_roc_chr.append(header2)

header_two = [
    "Турнир",
    "ID команды",
    "Название",
    "Сумма",
] + q_keys_two
roc_chr.append(header_two)

ARM_ID = 12169
CHR_ID = 11749
ROC_ID = 11756

for_three_results = []
for_two_results = []

for team_res in results[ARM_ID]:
    dct = {
        "tournament": "ЧА",
        "team_id": team_res["team_id"],
        "team_name": team_res["team_name"]
    }
    q_num_to_res = team_res["q_num_to_res"]
    dct["three_team_results"] = [
        (1 if q_num_to_res[x] == "1" else 0)
        for x in q_keys
    ]
    dct["three_total"] = sum(dct["three_team_results"])
    for_three_results.append(dct)
for id_ in (CHR_ID, ROC_ID):
    for team_res in results[id_]:
        dct = {
            "tournament": "ЧР" if id_ == CHR_ID else "РОК",
            "team_id": team_res["team_id"],
            "team_name": team_res["team_name"]
        }
        q_num_to_res = team_res["q_num_to_res"]
        dct["three_team_results"] = [
            (1 if q_num_to_res[am_ru[x]] == "1" else 0)
            for x in q_keys
        ]
        dct["three_total"] = sum(dct["three_team_results"])
        dct["two_team_results"] = [
            (1 if q_num_to_res[x] == "1" else 0)
            for x in q_keys_two
        ]
        dct["two_total"] = sum(dct["two_team_results"])
        for_three_results.append(dct)
        for_two_results.append(dct)

for_three_results_sorted = sorted(for_three_results, key=lambda x: x["three_total"], reverse=True)
for dct in for_three_results_sorted:
    row = [
        dct["tournament"],
        dct["team_id"],
        dct["team_name"],
        dct["three_total"],
        "",
    ] + dct["three_team_results"]
    am_roc_chr.append(row)
for_two_results_sorted = sorted(for_two_results, key=lambda x: x["two_total"], reverse=True)
for dct in for_two_results_sorted:
    row = [
        dct["tournament"],
        dct["team_id"],
        dct["team_name"],
        dct["two_total"]
    ] + dct["two_team_results"]
    roc_chr.append(row)

wb.save("merged_results.xlsx")